### **Drzewo trójmianowe**

Poniższa funkcja była próbą (niestety nieudaną) wyznaczenia wartości opcji *knock-and-out* przy założeniu modelu *Uncertain Volatility* i użycia drzewa trójmianowego.

Jako, że była nieudana to nie opisuję szczegółów...

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy
import csv
import pandas as pd

In [ ]:
def ko_option_trinomial_tree_uvm(
    option_type: str,
    option_barrier: float,
    option_strike: float,
    time_to_maturity: float,
    S0: float,
    volatility: float,
    volmin: float,
    volmax: float,
    risk_free_rate: float,
    steps: int    
):
    
    r = risk_free_rate
    time_step = time_to_maturity / steps
    
    if time_step >= 2*volatility**2 / r**2:
        raise ValueError('Warunek dla p niespełniony')
    
    u = np.exp(volatility * np.sqrt(2 * time_step))
    
    pu_max = ((np.exp(r * time_step / 2) - np.exp( - volmax * np.sqrt(time_step / 2))) / (np.exp(volmax * np.sqrt(time_step / 2)) - np.exp( - volmax * np.sqrt(time_step / 2))))**2
    pd_max = ((np.exp(volmax * np.sqrt(time_step / 2)) - np.exp(r * time_step / 2)) / (np.exp(volmax * np.sqrt(time_step / 2)) - np.exp( - volmax * np.sqrt(time_step / 2))))**2
    pm_max = 1 - pu_max - pd_max
    
    pu_min = ((np.exp(r * time_step / 2) - np.exp( - volmin * np.sqrt(time_step / 2))) / (np.exp(volmin * np.sqrt(time_step / 2)) - np.exp( - volmin * np.sqrt(time_step / 2))))**2
    pd_min = ((np.exp(volmin * np.sqrt(time_step / 2)) - np.exp(r * time_step / 2)) / (np.exp(volmin * np.sqrt(time_step / 2)) - np.exp( - volmin * np.sqrt(time_step / 2))))**2
    pm_min = 1 - pu_min - pd_min
    
    discount_factor = np.exp( - r * time_step)   
    
    option_values_lattice = np.zeros((2*steps + 1)*(steps + 1)).reshape(2*steps + 1, steps + 1)
    price_lattice = np.ones((2*steps + 1)*(steps + 1)).reshape(2*steps + 1, steps + 1)
    
    price_lattice[0, 0] = S0
    for i in range(steps):
        price_lattice[0:(2*(i+1) + 1), i + 1] = S0 * u ** np.array(list(range(i + 1, -(i + 1) - 1, -1)))
   
    if option_type == 'call':
        barrier_memory_lattice = (price_lattice <= option_barrier)
    elif option_type == 'put':
        barrier_memory_lattice = (price_lattice >= option_barrier)
    else:
        raise NotImplementedError(
            f"{option_type} is not a valid option type (only call/put permited)"
        )

    for i in range(steps, -1, -1):
        for j in range(0, 2*i + 1):
            if i == steps:
                if option_type == 'call':
                    option_values_lattice[j, i] = max(price_lattice[j, i] - option_strike, 0) * barrier_memory_lattice[j, i]
                elif option_type == 'put':
                    option_values_lattice[j, i] = max(option_strike - price_lattice[j, i], 0) * barrier_memory_lattice[j, i]
                else:
                    raise NotImplementedError(
                        f"{option_type} is not a valid option type (only call/put permited)"
                    )
            else:               
                gamma = (option_values_lattice[j, i + 1] - 2 * option_values_lattice[j + 1, i + 1] + option_values_lattice[j + 2, i + 1])/((price_lattice[j, i + 1] - price_lattice[j + 1, i + 1])*(price_lattice[j + 1, i + 1] - price_lattice[j + 2, i + 1]))
                
                if gamma >= 0:                    
                    option_value = option_values_lattice[j, i + 1] * pu_max + option_values_lattice[j + 1, i + 1] * pm_max + option_values_lattice[j + 2, i + 1] * pd_max
                else: 
                    option_value = option_values_lattice[j, i + 1] * pu_min + option_values_lattice[j + 1, i + 1] * pm_min + option_values_lattice[j + 2, i + 1] * pd_min
                    
                option_values_lattice[j, i] = option_value * barrier_memory_lattice[j, i] * discount_factor
                                       
    option_price = option_values_lattice[0, 0]

    return(option_price)  

In [191]:
x = ko_option_trinomial_tree_uvm(
    option_type = 'call',
    option_barrier = 3600,
    option_strike = 3200,
    time_to_maturity = 1,
    S0 = 3200,
    volatility = 0.2,
    volmin = 0.2,
    volmax = 0.3,
    risk_free_rate = 0.05,
    steps = 10000
)

In [193]:
x

np.float64(8.403275786581883)